In [13]:
import pandas as pd
import numpy as np

# ---- load spot price
price = pd.read_parquet("data/module1_spot_v0.parquet")
price["ts"] = pd.to_datetime(price["ts"], utc=True)
price = price.set_index("ts").sort_index()

# ---- load events
events = pd.read_parquet("data/btc_events.parquet")

# Use THESE for time logic
events["t0_ts"] = pd.to_datetime(events["t0_ts"], utc=True)
events["t1_ts"] = pd.to_datetime(events["t1_ts"], utc=True)


events

,t0,t1,label,ret,sigma,primary_signal,meta_label,uniqueness,sample_weight,primary_pred,...,t1_ts,ts,p0,p1,p2,p3,p4,vol_pred,vol_regime,trend_state
0,21,26,-1,-0.082237,0.051541,1.0,0,0.408333,1.716137,1.0,...,2017-09-13 00:00:00+00:00,2017-09-08,0.368428,0.174742,0.185914,0.178164,0.092752,0.045075,high,up
1,22,26,-1,-0.076619,0.048957,1.0,0,0.290000,1.218807,1.0,...,2017-09-13 00:00:00+00:00,2017-09-09,0.063819,0.106668,0.457580,0.264642,0.107291,0.045075,high,up
2,23,27,-1,-0.258653,0.047303,1.0,0,0.285185,1.198572,1.0,...,2017-09-14 00:00:00+00:00,2017-09-10,0.187509,0.245987,0.230825,0.167934,0.167745,0.045305,high,up
3,24,26,-1,-0.064729,0.045503,1.0,0,0.205556,0.863906,1.0,...,2017-09-13 00:00:00+00:00,2017-09-11,0.211726,0.198270,0.267075,0.236859,0.086070,0.045397,high,up
4,25,26,-1,-0.054039,0.043313,-1.0,1,0.183333,0.770510,1.0,...,2017-09-13 00:00:00+00:00,2017-09-12,0.209975,0.150833,0.432476,0.129632,0.077085,0.045355,high,up
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3010,3031,3035,1,0.036802,0.027984,-1.0,0,0.235714,0.990656,1.0,...,2025-12-09 00:00:00+00:00,2025-12-05,0.016994,0.483694,0.437263,0.045924,0.016125,0.026639,low,up
3011,3032,3035,1,0.037846,0.026623,1.0,1,0.233333,0.980649,1.0,...,2025-12-09 00:00:00+00:00,2025-12-06,0.005442,0.442768,0.335275,0.164360,0.052155,0.026541,low,up
3012,3033,3041,-1,-0.044833,0.025750,1.0,0,0.362500,1.523509,1.0,...,2025-12-15 00:00:00+00:00,2025-12-07,0.014930,0.604662,0.327553,0.034013,0.018841,0.025947,low,up
3013,3034,3040,-1,-0.027542,0.024520,1.0,0,0.230000,0.966640,1.0,...,2025-12-14 00:00:00+00:00,2025-12-08,0.008611,0.747666,0.212829,0.021306,0.009588,0.025947,low,up


In [3]:
def generate_walk_forward_splits(start, end, train_years=3, test_months=6):
    splits = []
    train_delta = pd.DateOffset(years=train_years)
    test_delta = pd.DateOffset(months=test_months)

    train_start = start
    train_end = train_start + train_delta

    while train_end + test_delta <= end:
        test_start = train_end
        test_end = train_end + test_delta

        splits.append((train_start, train_end, test_start, test_end))
        train_end = test_end

    return splits


In [14]:
START = price.index.min()
END   = price.index.max()
print(START,END)
splits = generate_walk_forward_splits(START, END)

ledger = []

for fold_id, (_, _, te_start, te_end) in enumerate(splits):

    test_events = events[
        (events.t0_ts >= te_start) &
        (events.t0_ts <  te_end)
    ]

    for _, ev in test_events.iterrows():

        if ev.meta_label != 1:
            continue

        entry_time = ev.t0_ts + pd.Timedelta(days=1)
        exit_time  = ev.t1_ts

        if entry_time not in price.index or exit_time not in price.index:
            continue

        entry_px = price.loc[entry_time, "open"]
        exit_px  = price.loc[exit_time, "close"]

        ret = ev.primary_pred * (exit_px - entry_px) / entry_px

        ledger.append({
            "event_id": ev.name,
            "fold_id": fold_id,
            "t0_ts": ev.t0_ts,
            "t1_ts": ev.t1_ts,
            "entry_time": entry_time,
            "exit_time": exit_time,
            "direction": ev.primary_pred,
            "ret": ret
        })


2017-08-18 00:00:00+00:00 2025-12-29 00:00:00+00:00


In [15]:
ledger

[{'event_id': 1078,
  'fold_id': 0,
  't0_ts': Timestamp('2020-08-21 00:00:00+0000', tz='UTC'),
  't1_ts': Timestamp('2020-09-01 00:00:00+0000', tz='UTC'),
  'entry_time': Timestamp('2020-08-22 00:00:00+0000', tz='UTC'),
  'exit_time': Timestamp('2020-09-01 00:00:00+0000', tz='UTC'),
  'direction': 1.0,
  'ret': np.float64(0.03388537042449069)},
 {'event_id': 1082,
  'fold_id': 0,
  't0_ts': Timestamp('2020-08-25 00:00:00+0000', tz='UTC'),
  't1_ts': Timestamp('2020-08-30 00:00:00+0000', tz='UTC'),
  'entry_time': Timestamp('2020-08-26 00:00:00+0000', tz='UTC'),
  'exit_time': Timestamp('2020-08-30 00:00:00+0000', tz='UTC'),
  'direction': 1.0,
  'ret': np.float64(0.0346991894628402)},
 {'event_id': 1087,
  'fold_id': 0,
  't0_ts': Timestamp('2020-08-30 00:00:00+0000', tz='UTC'),
  't1_ts': Timestamp('2020-09-02 00:00:00+0000', tz='UTC'),
  'entry_time': Timestamp('2020-08-31 00:00:00+0000', tz='UTC'),
  'exit_time': Timestamp('2020-09-02 00:00:00+0000', tz='UTC'),
  'direction': 1.0,


In [16]:
ledger = pd.DataFrame(ledger).sort_values("exit_time").reset_index(drop=True)

ledger["cum_pnl"] = ledger["ret"].cumsum()
ledger["peak"] = ledger["cum_pnl"].cummax()
ledger["drawdown"] = ledger["cum_pnl"] - ledger["peak"]

mean_ret = ledger.ret.mean()
hit_rate = (ledger.ret > 0).mean()
max_dd = ledger.drawdown.min()

sharpe = (
    mean_ret / ledger.ret.std() * np.sqrt(252)
    if ledger.ret.std() > 0 else np.nan
)

print("Trades:", len(ledger))
print("Mean return:", mean_ret)
print("Hit rate:", hit_rate)
print("Sharpe:", sharpe)
print("Max drawdown:", max_dd)


Trades: 911
Mean return: 0.010236202065906874
Hit rate: 0.6114160263446762
Sharpe: 3.0040709015201505
Max drawdown: -5.5538377029167725


In [17]:
ledger.groupby("fold_id")["ret"].agg(["count", "mean", "sum"])


,count,mean,sum
fold_id,,,
0,113,0.051766,5.849606
1,86,-0.013888,-1.194371
2,91,-0.010211,-0.929192
3,80,-0.015720,-1.257611
4,84,0.003869,0.324963
5,84,0.005676,0.476791
6,105,0.029421,3.089177
7,93,0.004224,0.392796
8,90,0.021346,1.921119


In [18]:
ledger.groupby("exit_time").size().describe()


count    394.000000
mean       2.312183
std        1.937051
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max       16.000000
dtype: float64

In [19]:
cost = 0.002  # 20 bps
ledger["ret_net"] = ledger["ret"] - cost

ledger["ret_net"].mean()
ledger["ret_net"].cumsum().iloc[-1]
ledger["ret_net"].min()


np.float64(-0.15928620640075045)

In [20]:
# ----------------------------------
# Save baseline ledger
# ----------------------------------
ledger = ledger.copy()

# Enforce types (important)
ledger["fold_id"] = ledger["fold_id"].astype(int)
ledger["exit_time"] = pd.to_datetime(ledger["exit_time"], utc=True)
ledger["entry_time"] = pd.to_datetime(ledger["entry_time"], utc=True)

ledger.to_parquet(
    "data/ledger_baseline.parquet",
    engine="pyarrow",
    index=False
)

print("Saved:", ledger.shape)


Saved: (911, 12)


In [21]:
# Required columns check
required_cols = {
    "t0_ts", "t1_ts",
    "primary_pred", "primary_prob",
    "meta_label", "sigma",
    "vol_regime", "trend_state",
    "fold_id"
}

missing = required_cols - set(ledger.columns)
assert not missing, f"Missing columns: {missing}"



KeyboardInterrupt

